In [4]:
import torch
from diffusion.flows.prob_paths import GaussianCondProbPath
from diffusion.training.trainer_fmm import FMMTrainer
from diffusion.sampleables.sampleable_mnist import MNISTSampleable
from diffusion.backbones.res_unet import BiTimeResUnet

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [6]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(p_data=sampeable, p_simple_shape=sampeable.shape).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable, p_simple_shape=sampeable.shape
).to(device)

backbone = BiTimeResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FMMTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [7]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-12-22 13:23:34,101 - flow-matching - INFO - Training model with size: 2.436 MiB
Epoch 0/15:  75%|███████▍  | 374/500 [07:33<02:32,  1.21s/it, train_loss=26.637129]  


KeyboardInterrupt: 

In [ ]:
torch.save(backbone.state_dict(), "./models/backbone_flow_bili.pt")